#Libraries

In [1]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from sklearn.model_selection import train_test_split



#Check the masks

In [2]:
# paths
ct_dir = r"D:\My Projects\Brain Dataset\Brain Data Prepared\computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1\ct_scans"
mask_dir = r"D:\My Projects\Brain Dataset\Brain Data Prepared\computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1\masks"

output_dir = "/content/splits"
os.makedirs(output_dir, exist_ok=True)

scan_ids = []
labels = []

# ---- STEP 1: build scan-level labels ----
for file_name in sorted(os.listdir(mask_dir)):

    if not file_name.endswith(".nii"):
        continue

    mask_path = os.path.join(mask_dir, file_name)

    mask = nib.load(mask_path).get_fdata()

    # if ANY voxel > 0 → hemorrhage
    label = 1 if np.any(mask > 0) else 0

    scan_id = file_name.replace(".nii", "")

    scan_ids.append(scan_id)
    labels.append(label)

df = pd.DataFrame({
    "scan_id": scan_ids,
    "label": labels
})

print("Scan-level distribution:")
print(df["label"].value_counts())


Scan-level distribution:
label
0    39
1    36
Name: count, dtype: int64


#Splitting

In [3]:
# ---- STEP 2: split (train 80 / temp 20) ----
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [4]:
# ---- STEP 3: split temp → val/test (10/10) ----
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

In [5]:
print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nVal distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

# ---- STEP 4: save ----
train_df.to_csv(os.path.join(output_dir, "train_scans.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "val_scans.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test_scans.csv"), index=False)

print("\nSaved split files in:", output_dir)


Train distribution:
label
0    31
1    29
Name: count, dtype: int64

Val distribution:
label
0    4
1    3
Name: count, dtype: int64

Test distribution:
label
1    4
0    4
Name: count, dtype: int64

Saved split files in: /content/splits


#Save the CSV files

In [6]:
import os

# destination folder
drive_output_dir = "D:\\My Projects\\Brain Dataset\\Brain Data Prepared\\splits"
os.makedirs(drive_output_dir, exist_ok=True)

# save CSVs to Drive
train_df.to_csv(os.path.join(drive_output_dir, "train_scans.csv"), index=False)
val_df.to_csv(os.path.join(drive_output_dir, "val_scans.csv"), index=False)
test_df.to_csv(os.path.join(drive_output_dir, "test_scans.csv"), index=False)

print("✅ CSV files saved to:", drive_output_dir)

✅ CSV files saved to: D:\My Projects\Brain Dataset\Brain Data Prepared\splits
